# Fruit Quality Classification
### Quality Grading (`good`, `medium`, `bad`) Benchmark for Selected Fruit
This study evaluates six computer vision architectures on fruit quality assessment:
- Custom Quality CNN
- MobileNetV3-Large
- YOLOv8 Classifier
- YOLO26 Classifier
- EfficientNet-B0
- ResNet-50

**Target Quality Classes:** `good`, `medium`, `bad`

**Pipeline Outline:**
1. Environment Setup and Reproducibility
2. Fruit Category Selection and Quality Scan
3. Stratified Partitioning (75% Train, 15% Validation, 10% Test)
4. Quality-Preserving Data Augmentation
5. Training and Evaluation Infrastructure
6. Architectural Evaluation (Models 1 through 6)
7. Final Performance Comparison


## 1. Environment Setup and Reproducibility


In [ ]:
# Dependencies installation
!pip install -q ultralytics scikit-learn seaborn matplotlib pandas torchvision torch optuna tqdm

import os
import sys
import math
import random
import shutil
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize

from ultralytics import YOLO

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Execution environment detection
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ENV_NAME = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local Machine")

print(f"Environment: {ENV_NAME}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
    torch.backends.cudnn.benchmark = True
else:
    print("Warning: CUDA is unavailable. Running on CPU.")
    device = torch.device("cpu")


In [ ]:
# Seed configuration for reproducibility
SEED = 42

def seed_everything(seed=42):
    """Seed Python, NumPy, and PyTorch for deterministic execution."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print(f"Random seed locked: {SEED}")


In [ ]:
# Output directory initialization
OUTPUT_QUALITY_DIR = Path("outputs_quality")
MODELS_Q_DIR = OUTPUT_QUALITY_DIR / "models"
PLOTS_Q_DIR = OUTPUT_QUALITY_DIR / "plots"
REPORTS_Q_DIR = OUTPUT_QUALITY_DIR / "reports"

for p in [OUTPUT_QUALITY_DIR, MODELS_Q_DIR, PLOTS_Q_DIR, REPORTS_Q_DIR]:
    p.mkdir(parents=True, exist_ok=True)


### Dataset Download & Extract


In [ ]:
!python -m pip install --upgrade gdown -q
!gdown "https://drive.google.com/file/d/1RNFxIXg9lvxGuI-LwA6vh_ZsMnwrB_i8/view?usp=drive_link" -q
!mkdir -p original_dataset && cd "original_dataset" && tar -xf /content/fruit_dataset.tar


## 2. Fruit Selection and Targeted Scanning
Isolating the selected fruit category for quality grading.


In [ ]:
#@title Select Fruit for Quality Classification { run: "auto" }
SELECTED_FRUIT = "guava" #@param ["banana", "dragonfruit", "jackfruit", "lemon", "mango", "pineapple", "star_fruit", "custard_apple", "guava", "jujube", "lychee", "papaya", "sapodilla"] {type:"string"}

QUALITY_CLASSES = ['good', 'medium', 'bad']
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def locate_quality_dataset_dir():
    candidates = [
        Path("original_dataset"),
        Path("/content/original_dataset"),
        Path("/kaggle/working/original_dataset")
    ]
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("original_dataset"):
            if p.is_dir():
                candidates.insert(0, p)
        for p in Path("/kaggle/input").glob("*"):
            if p.is_dir() and (p / SELECTED_FRUIT).exists():
                candidates.insert(0, p)

    for c in candidates:
        if c.exists() and (c / SELECTED_FRUIT).exists():
            return c
    return Path("original_dataset")

ORIGINAL_DATASET_DIR = locate_quality_dataset_dir()
print(f"Target fruit: {SELECTED_FRUIT.upper()}")


In [ ]:
def scan_fruit_quality(base_dir: Path, fruit_name: str):
    fruit_dir = base_dir / fruit_name
    if not fruit_dir.exists():
        raise FileNotFoundError(f"Directory '{fruit_dir}' not found.")

    records = []
    for quality in QUALITY_CLASSES:
        q_dir = fruit_dir / quality
        if q_dir.exists():
            imgs = [f for f in q_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS and f.is_file()]
            for img in imgs:
                records.append({
                    "fruit": fruit_name,
                    "quality": quality,
                    "file_name": img.name,
                    "path": str(img)
                })

    return pd.DataFrame(records)

df_quality = scan_fruit_quality(ORIGINAL_DATASET_DIR, SELECTED_FRUIT)
print(f"Total quality samples for '{SELECTED_FRUIT}': {len(df_quality)}")
df_quality.head()


In [ ]:
# Quality distribution visualization
if not df_quality.empty:
    quality_counts = df_quality['quality'].value_counts()
    display(quality_counts)
    quality_counts.to_csv(REPORTS_Q_DIR / f"{SELECTED_FRUIT}_quality_distribution.csv")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(x=quality_counts.index, y=quality_counts.values, palette="Set1", ax=axes[0])
    axes[0].set_title(f"Sample Count per Quality Grade ({SELECTED_FRUIT.capitalize()})")
    axes[0].set_xlabel("Quality Grade")
    axes[0].set_ylabel("Count")

    axes[1].pie(quality_counts.values, labels=quality_counts.index, autopct='%1.1f%%', colors=sns.color_palette("Set1"))
    axes[1].set_title(f"Quality Proportion ({SELECTED_FRUIT.capitalize()})")
    plt.tight_layout()
    plt.savefig(PLOTS_Q_DIR / f"{SELECTED_FRUIT}_distribution.png", dpi=300)
    plt.show()


## 3. Stratified Partitioning (75% Train, 15% Val, 10% Test)
Stratified splitting across the three quality classes (`good`, `medium`, `bad`).


In [ ]:
TARGET_QUALITY_DIR = Path("data_quality")

def prepare_stratified_quality_split(df: pd.DataFrame, target_base: Path, train_ratio=0.75, val_ratio=0.15, test_ratio=0.10):
    assert math.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Split ratios must sum to 1.0"

    if target_base.exists():
        shutil.rmtree(target_base)

    for split in ['train', 'val', 'test']:
        for q in QUALITY_CLASSES:
            (target_base / split / q).mkdir(parents=True, exist_ok=True)

    temp_val_ratio = val_ratio / (train_ratio + val_ratio)

    train_val_df, test_df = train_test_split(
        df,
        test_size=test_ratio,
        stratify=df['quality'],
        random_state=SEED
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=temp_val_ratio,
        stratify=train_val_df['quality'],
        random_state=SEED
    )

    splits = {'train': train_df, 'val': val_df, 'test': test_df}

    print(f"Quality partitions:")
    print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

    copy_records = []
    for split_name, split_data in splits.items():
        pbar = tqdm(split_data.iterrows(), total=len(split_data), desc=f"Preparing {split_name} set", leave=False)
        for _, row in pbar:
            src_path = Path(row['path'])
            q_class = row['quality']
            dst_path = target_base / split_name / q_class / src_path.name
            shutil.copy2(src_path, dst_path)
            copy_records.append({
                "split": split_name,
                "quality": q_class,
                "file": src_path.name,
                "path": str(dst_path)
            })

    df_split_res = pd.DataFrame(copy_records)
    return df_split_res

df_q_splits = prepare_stratified_quality_split(df_quality, TARGET_QUALITY_DIR)


In [ ]:
# Quality partition verification
split_table = pd.crosstab(df_q_splits['quality'], df_q_splits['split'])
split_table['Total'] = split_table.sum(axis=1)
split_table['Train%'] = (split_table['train'] / split_table['Total'] * 100).round(1)
split_table['Val%'] = (split_table['val'] / split_table['Total'] * 100).round(1)
split_table['Test%'] = (split_table['test'] / split_table['Total'] * 100).round(1)
display(split_table)
split_table.to_csv(REPORTS_Q_DIR / f"{SELECTED_FRUIT}_stratified_split_verification.csv")


## 4. Quality-Preserving Data Augmentation
Augmentations designed for defect preservation, avoiding extreme photometric distortions that could obscure subtle surface defects or ripeness cues.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

quality_train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    NORMALIZE
])

quality_val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

train_q_dataset = datasets.ImageFolder(root="data_quality/train", transform=quality_train_transforms)
val_q_dataset = datasets.ImageFolder(root="data_quality/val", transform=quality_val_test_transforms)
test_q_dataset = datasets.ImageFolder(root="data_quality/test", transform=quality_val_test_transforms)

train_q_loader = DataLoader(
    train_q_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_q_loader = DataLoader(
    val_q_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

test_q_loader = DataLoader(
    test_q_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

quality_class_names = train_q_dataset.classes
NUM_QUALITY_CLASSES = len(quality_class_names)
print(f"Target quality classes ({NUM_QUALITY_CLASSES}): {quality_class_names}")


## 5. Training and Evaluation Infrastructure


In [ ]:
all_quality_results = {}

def train_quality_pytorch(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=15, patience=4, model_name="model"):
    slug = model_name.lower().replace(" ", "_").replace("-", "_")
    checkpoint_path = MODELS_Q_DIR / f"best_{slug}.pth"
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }
    best_val_f1 = 0.0
    early_stop_cnt = 0
    start_t = time.time()
    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    for epoch in range(num_epochs):
        ep_start = time.time()
        model.train()
        r_loss, r_corr, total_t = 0.0, 0, 0
        train_preds, train_targets = [], []
        train_pbar = tqdm(train_loader, desc=f"{model_name} [Epoch {epoch+1:02d}/{num_epochs:02d}] Train", leave=False)
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            _, preds = torch.max(outputs, 1)
            r_loss += loss.item() * inputs.size(0)
            r_corr += torch.sum(preds == labels.data)
            total_t += labels.size(0)

            train_preds.extend(preds.detach().cpu().numpy())
            train_targets.extend(labels.detach().cpu().numpy())

            train_pbar.set_postfix({'loss': f"{r_loss/total_t:.4f}", 'acc': f"{(r_corr.double()/total_t).item()*100:.2f}%"})

        if scheduler:
            scheduler.step()
        train_loss = r_loss / total_t if total_t > 0 else 0
        train_acc = (r_corr.double() / total_t).item() if total_t > 0 else 0
        train_f1 = f1_score(train_targets, train_preds, average='macro', zero_division=0)

        model.eval()
        vr_loss, vr_corr, total_v = 0.0, 0, 0
        val_preds, val_targets = [], []
        val_pbar = tqdm(val_loader, desc=f"{model_name} [Epoch {epoch+1:02d}/{num_epochs:02d}] Val", leave=False)
        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                vr_loss += loss.item() * inputs.size(0)
                vr_corr += torch.sum(preds == labels.data)
                total_v += labels.size(0)

                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

                val_pbar.set_postfix({'val_loss': f"{vr_loss/total_v:.4f}", 'val_acc': f"{(vr_corr.double()/total_v).item()*100:.2f}%"})

        val_loss = vr_loss / total_v if total_v > 0 else 0
        val_acc = (vr_corr.double() / total_v).item() if total_v > 0 else 0
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        ep_time = time.time() - ep_start
        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} [{ep_time:.1f}s] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% F1: {train_f1:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% F1: {val_f1:.4f}")

        # Checkpointing based on validation Macro F1 score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            early_stop_cnt = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= patience:
                print(f"Early stopping triggered at epoch {epoch+1} (Best Val Macro F1: {best_val_f1:.4f}).")
                break

    time_elapsed = time.time() - start_t
    print(f"Completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s. Best validation Macro F1: {best_val_f1:.4f}")

    # 3-Panel Learning Curves Plot: Loss, Accuracy, and Macro F1
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    # 1. Loss Curve
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='royalblue')
    axes[0].plot(epochs, history['val_loss'], 's-', label='Val Loss', color='crimson')
    axes[0].set_title(f"{model_name}: Loss vs Epochs")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    # 2. Accuracy Curve
    axes[1].plot(epochs, [a * 100 for a in history['train_acc']], 'o-', label='Train Acc', color='royalblue')
    axes[1].plot(epochs, [a * 100 for a in history['val_acc']], 's-', label='Val Acc', color='crimson')
    axes[1].set_title(f"{model_name}: Accuracy vs Epochs")
    axes[1].set_xlabel("Epochs")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    # 3. Macro F1 Curve
    axes[2].plot(epochs, history['train_f1'], 'o-', label='Train Macro F1', color='royalblue')
    axes[2].plot(epochs, history['val_f1'], 's-', label='Val Macro F1', color='crimson')
    axes[2].set_title(f"{model_name}: Macro F1 vs Epochs")
    axes[2].set_xlabel("Epochs")
    axes[2].set_ylabel("Macro F1-Score")
    axes[2].set_ylim([0, 1.05])
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(PLOTS_Q_DIR / f"{slug}_learning_curves.png", dpi=300)
    plt.show()

    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path))
    return model, history


In [ ]:
def evaluate_and_record_quality(y_true, y_pred, y_prob, model_name, class_names, results_dict):
    slug = model_name.lower().replace(" ", "_").replace("-", "_")

    print(f"\n=======================================================")
    print(f"      {model_name.upper()} QUALITY EVALUATION ({SELECTED_FRUIT.upper()})")
    print(f"=======================================================")
    rep_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print(rep_text)

    with open(REPORTS_Q_DIR / f"{slug}_classification_report.txt", "w") as f:
        f.write(rep_text)
    pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, output_dict=True)).transpose().to_csv(REPORTS_Q_DIR / f"{slug}_classification_report.csv")

    acc = np.mean(y_true == y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    np.savez_compressed(
        REPORTS_Q_DIR / f"{slug}_predictions.npz",
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob
    )

    results_dict[model_name] = {
        'Fruit': SELECTED_FRUIT,
        'Accuracy': acc,
        'Macro F1': macro_f1,
        'Weighted F1': weighted_f1
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f"{model_name} - Confusion Matrix ({SELECTED_FRUIT.capitalize()})")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fpr, tpr, roc_auc = dict(), dict(), dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["macro"], tpr["macro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    colors = ['green', 'goldenrod', 'crimson']
    for i, color in zip(range(n_classes), colors):
        axes[1].plot(fpr[i], tpr[i], color=color, lw=1.5, label=f'{class_names[i].upper()} (AUC = {roc_auc[i]:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1.2)
    axes[1].set_title(f"{model_name} - ROC Curves")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.savefig(PLOTS_Q_DIR / f"{slug}_confusion_matrix_and_roc.png", dpi=300)
    plt.show()

def eval_quality_model(model, loader):
    model.eval()
    all_p, all_l, all_prob = [], [], []
    use_amp = (device.type == 'cuda')
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Inference", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_prob.append(probs.cpu().numpy())
            all_p.append(preds.cpu().numpy())
            all_l.append(labels.numpy())
    return np.concatenate(all_l), np.concatenate(all_p), np.concatenate(all_prob)

def cleanup_gpu_memory(model_var_names):
    for name in model_var_names:
        if name in globals():
            del globals()[name]
    torch.cuda.empty_cache()
    gc.collect()


In [ ]:
def plot_yolo_learning_curves(run_dir, model_name, output_plots_dir):
    """Extract and plot training/validation metrics for YOLO from results.csv."""
    slug = model_name.lower().replace(" ", "_").replace("-", "_")
    csv_path = Path(run_dir) / "results.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        epochs = df['epoch'] if 'epoch' in df.columns else range(1, len(df) + 1)

        has_train_loss = 'train/loss' in df.columns
        has_val_loss = 'val/loss' in df.columns
        has_acc = 'metrics/accuracy_top1' in df.columns

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        if has_train_loss and has_val_loss:
            axes[0].plot(epochs, df['train/loss'], 'o-', label='Train Loss', color='royalblue')
            axes[0].plot(epochs, df['val/loss'], 's-', label='Val Loss', color='crimson')
            axes[0].set_title(f"{model_name}: Loss vs Epochs")
            axes[0].set_xlabel("Epochs")
            axes[0].set_ylabel("Loss")
            axes[0].legend()

        if has_acc:
            axes[1].plot(epochs, df['metrics/accuracy_top1'] * 100, 's-', label='Val Top-1 Acc', color='crimson')
            axes[1].set_title(f"{model_name}: Top-1 Accuracy vs Epochs")
            axes[1].set_xlabel("Epochs")
            axes[1].set_ylabel("Accuracy (%)")
            axes[1].legend()

        plt.tight_layout()
        plt.savefig(Path(output_plots_dir) / f"{slug}_learning_curves.png", dpi=300)
        plt.show()

    yolo_png = Path(run_dir) / "results.png"
    if yolo_png.exists():
        shutil.copy2(yolo_png, Path(output_plots_dir) / f"{slug}_ultralytics_results.png")



---
## Model 1: Custom Quality CNN
4-stage convolutional neural network for 3-class quality grading.



In [ ]:
class SmallQualityCNN(nn.Module):
    def __init__(self, num_classes=3, dropout_rate=0.3):
        super(SmallQualityCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.head(self.global_pool(self.features(x)))

# Screening
best_q_lr, best_q_wd, best_q_drop = 1e-3, 1e-4, 0.3
for cand in [{"lr": 1e-3, "weight_decay": 1e-4, "dropout": 0.3}, {"lr": 5e-4, "weight_decay": 1e-3, "dropout": 0.2}]:
    t_model = SmallQualityCNN(num_classes=NUM_QUALITY_CLASSES, dropout_rate=cand['dropout']).to(device)
    opt = optim.AdamW(t_model.parameters(), lr=cand['lr'], weight_decay=cand['weight_decay'])
    crit = nn.CrossEntropyLoss()
    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    t_model.train()
    for b_idx, (inputs, labels) in enumerate(train_q_loader):
        if b_idx >= 25:
            break
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            loss = crit(t_model(inputs), labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
del t_model, opt, scaler
torch.cuda.empty_cache()

# Training and evaluation
quality_cnn = SmallQualityCNN(num_classes=NUM_QUALITY_CLASSES, dropout_rate=best_q_drop).to(device)
criterion_q = nn.CrossEntropyLoss()
opt_q_cnn = optim.AdamW(quality_cnn.parameters(), lr=best_q_lr, weight_decay=best_q_wd)
sched_q_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_q_cnn, T_max=15)

quality_cnn, q_cnn_hist = train_quality_pytorch(
    quality_cnn, train_q_loader, val_q_loader, criterion_q, opt_q_cnn,
    scheduler=sched_q_cnn, num_epochs=25, patience=4, model_name="Small Custom CNN"
)

y_true_q_cnn, y_pred_q_cnn, y_prob_q_cnn = eval_quality_model(quality_cnn, test_q_loader)
evaluate_and_record_quality(y_true_q_cnn, y_pred_q_cnn, y_prob_q_cnn, "Small Custom CNN", quality_class_names, all_quality_results)

cleanup_gpu_memory(['quality_cnn', 'opt_q_cnn', 'sched_q_cnn', 'criterion_q'])


---
## Model 2: MobileNetV3-Large
Pretrained lightweight architecture fine-tuned for quality assessment.


In [ ]:
def build_quality_mobilenet(num_classes=3):
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

mb_q_model = build_quality_mobilenet(num_classes=NUM_QUALITY_CLASSES).to(device)
criterion_mb_q = nn.CrossEntropyLoss()
opt_mb_q = optim.AdamW(mb_q_model.parameters(), lr=2e-4, weight_decay=1e-4)
sched_mb_q = optim.lr_scheduler.CosineAnnealingLR(opt_mb_q, T_max=15)

mb_q_model, mb_q_hist = train_quality_pytorch(
    mb_q_model, train_q_loader, val_q_loader, criterion_mb_q, opt_mb_q,
    scheduler=sched_mb_q, num_epochs=15, patience=4, model_name="MobileNetV3-Large"
)

y_true_mb_q, y_pred_mb_q, y_prob_mb_q = eval_quality_model(mb_q_model, test_q_loader)
evaluate_and_record_quality(y_true_mb_q, y_pred_mb_q, y_prob_mb_q, "MobileNetV3-Large", quality_class_names, all_quality_results)

cleanup_gpu_memory(['mb_q_model', 'opt_mb_q', 'sched_mb_q', 'criterion_mb_q'])


---
## Model 3: YOLOv8 Classifier
Ultralytics YOLOv8 nano classification model (`yolov8n-cls.pt`).


In [ ]:
yolo8_q_model = YOLO("yolov8n-cls.pt")

yolo8_q_results = yolo8_q_model.train(
    data=str(Path("data_quality").resolve()),
    epochs=15,
    imgsz=224,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=SEED,
    workers=2,
    project="runs_quality_benchmark",
    name="yolov8n_quality",
    exist_ok=True
)

yolo8_q_weights = Path("runs_quality_benchmark/yolov8n_quality/weights/best.pt")
if yolo8_q_weights.exists():
    shutil.copy2(yolo8_q_weights, MODELS_Q_DIR / "best_yolo8n_quality.pt")

def evaluate_yolo_quality_preds(model, test_dir, class_names):
    test_path = Path(test_dir)
    labels, preds, probs = [], [], []
    c2i = {name: i for i, name in enumerate(class_names)}
    for cname in tqdm(class_names, desc="YOLO Inference", leave=False):
        cfolder = test_path / cname
        if not cfolder.exists():
            continue
        imgs = [f for f in cfolder.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        for img_p in imgs:
            res = model(str(img_p), verbose=False)[0]
            p_arr = res.probs.data.cpu().numpy()
            labels.append(c2i[cname])
            preds.append(int(np.argmax(p_arr)))
            probs.append(p_arr)
    return np.array(labels), np.array(preds), np.array(probs)

plot_yolo_learning_curves("runs_quality_benchmark/yolov8n_quality", "YOLO8n-cls", PLOTS_Q_DIR)

y_true_y8_q, y_pred_y8_q, y_prob_y8_q = evaluate_yolo_quality_preds(yolo8_q_model, "data_quality/test", quality_class_names)
evaluate_and_record_quality(y_true_y8_q, y_pred_y8_q, y_prob_y8_q, "YOLO8n-cls", quality_class_names, all_quality_results)

cleanup_gpu_memory(['yolo8_q_model', 'yolo8_q_results'])



---
## Model 4: YOLO26 Classifier
Ultralytics official YOLO26 classification architecture (`yolo26n-cls.pt`).


In [ ]:
yolo26_q_model = YOLO("yolo26n-cls.pt")

yolo26_q_results = yolo26_q_model.train(
    data=str(Path("data_quality").resolve()),
    epochs=15,
    imgsz=224,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=SEED,
    workers=2,
    project="runs_quality_benchmark",
    name="yolo26n_quality",
    exist_ok=True
)

yolo26_q_weights = Path("runs_quality_benchmark/yolo26n_quality/weights/best.pt")
if yolo26_q_weights.exists():
    shutil.copy2(yolo26_q_weights, MODELS_Q_DIR / "best_yolo26n_quality.pt")

plot_yolo_learning_curves("runs_quality_benchmark/yolo26n_quality", "YOLO26n-cls", PLOTS_Q_DIR)

y_true_y26_q, y_pred_y26_q, y_prob_y26_q = evaluate_yolo_quality_preds(yolo26_q_model, "data_quality/test", quality_class_names)
evaluate_and_record_quality(y_true_y26_q, y_pred_y26_q, y_prob_y26_q, "YOLO26n-cls", quality_class_names, all_quality_results)

cleanup_gpu_memory(['yolo26_q_model', 'yolo26_q_results'])



---
## Model 5: EfficientNet-B0
Compound scaling architecture fine-tuned for 3-class quality assessment.


In [ ]:
def build_quality_efficientnet(num_classes=3):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

eff_q_model = build_quality_efficientnet(num_classes=NUM_QUALITY_CLASSES).to(device)
criterion_eff_q = nn.CrossEntropyLoss()
opt_eff_q = optim.AdamW(eff_q_model.parameters(), lr=2e-4, weight_decay=1e-4)
sched_eff_q = optim.lr_scheduler.CosineAnnealingLR(opt_eff_q, T_max=15)

eff_q_model, eff_q_hist = train_quality_pytorch(
    eff_q_model, train_q_loader, val_q_loader, criterion_eff_q, opt_eff_q,
    scheduler=sched_eff_q, num_epochs=15, patience=4, model_name="EfficientNet-B0"
)

y_true_eff_q, y_pred_eff_q, y_prob_eff_q = eval_quality_model(eff_q_model, test_q_loader)
evaluate_and_record_quality(y_true_eff_q, y_pred_eff_q, y_prob_eff_q, "EfficientNet-B0", quality_class_names, all_quality_results)

cleanup_gpu_memory(['eff_q_model', 'opt_eff_q', 'sched_eff_q', 'criterion_eff_q'])


---
## Model 6: ResNet-50
50-layer deep residual network fine-tuned for quality assessment.


In [ ]:
def build_quality_resnet50(num_classes=3):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

resnet_q_model = build_quality_resnet50(num_classes=NUM_QUALITY_CLASSES).to(device)
criterion_res_q = nn.CrossEntropyLoss()
opt_res_q = optim.AdamW(resnet_q_model.parameters(), lr=1.5e-4, weight_decay=1e-4)
sched_res_q = optim.lr_scheduler.CosineAnnealingLR(opt_res_q, T_max=15)

resnet_q_model, resnet_q_hist = train_quality_pytorch(
    resnet_q_model, train_q_loader, val_q_loader, criterion_res_q, opt_res_q,
    scheduler=sched_res_q, num_epochs=15, patience=4, model_name="ResNet50"
)

y_true_res_q, y_pred_res_q, y_prob_res_q = eval_quality_model(resnet_q_model, test_q_loader)
evaluate_and_record_quality(y_true_res_q, y_pred_res_q, y_prob_res_q, "ResNet50", quality_class_names, all_quality_results)

cleanup_gpu_memory(['resnet_q_model', 'opt_res_q', 'sched_res_q', 'criterion_res_q'])


---
## 7. Comparative Benchmark and Results Summary


In [ ]:
summary_q_rows = []
for model_name, metrics in all_quality_results.items():
    summary_q_rows.append({
        "Model Architecture": model_name,
        "Fruit Evaluated": metrics['Fruit'],
        "Accuracy (%)": f"{metrics['Accuracy'] * 100:.2f}%",
        "Macro F1": f"{metrics['Macro F1']:.4f}",
        "Weighted F1": f"{metrics['Weighted F1']:.4f}"
    })

df_q_benchmark = pd.DataFrame(summary_q_rows)
display(df_q_benchmark)

q_csv_path = OUTPUT_QUALITY_DIR / "final_quality_benchmark_summary.csv"
df_q_benchmark.to_csv(q_csv_path, index=False)


In [ ]:
if not df_q_benchmark.empty:
    chart_q_data = pd.DataFrame([
        {
            "Model": name,
            "Accuracy (%)": metrics['Accuracy'] * 100,
            "Macro F1": metrics['Macro F1']
        }
        for name, metrics in all_quality_results.items()
    ])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.barplot(data=chart_q_data, x="Model", y="Accuracy (%)", palette="Blues_d", ax=axes[0])
    axes[0].set_title(f"Quality Accuracy Comparison ({SELECTED_FRUIT.capitalize()})", fontsize=13)
    axes[0].set_ylim([0, 100])
    axes[0].tick_params(axis='x', rotation=30)
    for p in axes[0].patches:
        axes[0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 7),
                         ha='center', va='center', color='white', fontweight='bold')

    sns.barplot(data=chart_q_data, x="Model", y="Macro F1", palette="Greens_d", ax=axes[1])
    axes[1].set_title(f"Quality Macro F1-Score Comparison ({SELECTED_FRUIT.capitalize()})", fontsize=13)
    axes[1].set_ylim([0, 1.0])
    axes[1].tick_params(axis='x', rotation=30)
    for p in axes[1].patches:
        axes[1].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height() - 0.07),
                         ha='center', va='center', color='white', fontweight='bold')

    plt.tight_layout()
    plt.savefig(PLOTS_Q_DIR / "final_quality_benchmark_comparison.png", dpi=300)
    plt.show()


## 8. Export Outputs


In [ ]:
# Compress outputs directory with dynamic fruit name
zip_filename = f"fruit_quality_outputs_{SELECTED_FRUIT}.zip"
shutil.make_archive(f"fruit_quality_outputs_{SELECTED_FRUIT}", 'zip', OUTPUT_QUALITY_DIR)
print(f"Outputs compressed to: {zip_filename}")

if 'google.colab' in sys.modules:
    try:
        from google.colab import files
        files.download(zip_filename)
    except Exception as e:
        print(f"Colab download skipped/error: {e}")
